# Training Pipeline

Notebook entry point for configuring and launching BiLSTM-GAT ST-GNN runs with W&B logging. Edit the `CONFIG` cell, then run the notebook top to bottom.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CONFIG = {
    # Data
    "year": 2019,
    "sentinel_dir": "/data/SentinelPV/energy_data/piemonte_energy_data/single_ups",
    "plant_mapping_path": "data/plant_mapping.csv",
    "energy_coords_path": "data/energy_with_coordinates.csv",
    "pvgis_path": "data/piedmont_pvgis_2019.nc",
    "upn_list": None,

    # Optional fleet filter, disabled by default
    "apply_outlier_filter": False,
    "qs_daytime_threshold": 0.30,
    "min_n_valid_daytime": 200,

    # Training
    "seeds": [42, 123, 2024],
    "seq_len": 24,
    "batch_size": 8,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 4,
    "graph_max_dist_km": 20.0,
    "n_epochs": 15,
    "max_steps_per_epoch": None,
    "early_stopping_patience": 5,
    "early_stopping_min_delta": 1e-4,
    "calibration_kpi": "none",
    "eta_max": 0.98,

    # Model
    "use_bilstm": True,
    "bilstm_pooling": "last",  # attn or last
    "use_gat": True,
    "d_model": 128,
    "gat_dim": 96,
    "gat_heads": 4,
    "gat_layers": 1,
    "dropout": 0.2,

    # Loss
    "lam": 0.1,
    "peak_alpha": 2.5,
    "peak_gamma": 2.0,
    "peak_loss_weight": 0.25,
    "under_penalty": 3.0,

    # Outputs and W&B
    "checkpoint_dir_base": "checkpoints/seq_len_24",
    "feature_set": "cloud_kt01_erbs",
    "use_wandb": True,
    "wandb_mode": "online",  # online, offline, disabled
    "wandb_entity": "albertopedalino-politecnico-di-torino",
    "wandb_project": "PhysiQ-PV",
    "wandb_tags": ["bilstm-gat", "erbs-dni-dhi", "multi_seed"],
}


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(ROOT))

from main import _filter_outlier_plants, _normalize_dataset
from train import train
from physiq_pv.data.dataset import N_FEATURES
from physiq_pv.data.load_kwp import load_kwp
from physiq_pv.data.sentinel_hourly_loader import load_sentinel_hourly, merge_with_weather


def project_path(value: str | Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


def jsonify(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {k: jsonify(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonify(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def run_name(cfg: dict, seed: int) -> str:
    return (
        f"{cfg['feature_set']}_f{N_FEATURES}_seq{cfg['seq_len']}"
        f"_a{cfg['peak_alpha']:g}_g{cfg['peak_gamma']:g}"
        f"_w{cfg['peak_loss_weight']:g}_pool{cfg['bilstm_pooling']}_seed{seed}"
    )


def checkpoint_dir(cfg: dict, seed: int) -> Path:
    base = project_path(cfg["checkpoint_dir_base"])
    return Path(f"{base}_pool{cfg['bilstm_pooling']}_seed{seed}")


if CONFIG["bilstm_pooling"] not in {"attn", "last"}:
    raise ValueError("bilstm_pooling must be 'attn' or 'last'")

if CONFIG["wandb_mode"]:
    os.environ["WANDB_MODE"] = CONFIG["wandb_mode"]

print(json.dumps(jsonify(CONFIG), indent=2))


In [ ]:
plant_mapping_path = project_path(CONFIG["plant_mapping_path"])
energy_coords_path = project_path(CONFIG["energy_coords_path"])
pvgis_path = project_path(CONFIG["pvgis_path"])

print("Loading Sentinel hourly data...")
ds = load_sentinel_hourly(
    sentinel_dir=CONFIG["sentinel_dir"],
    year=CONFIG["year"],
    plant_mapping_path=str(plant_mapping_path),
    energy_coords_path=str(energy_coords_path),
    upn_list=CONFIG["upn_list"],
)

print("Merging PVGIS weather data...")
ds = merge_with_weather(ds, pvgis_path=str(pvgis_path))
ds = _normalize_dataset(ds)
print(f"Dataset: {ds.sizes['plant']} plants x {ds.sizes['time']} timesteps")
print(f"Variables: {list(ds.data_vars.keys())}")

kwp = None
if plant_mapping_path.exists() and energy_coords_path.exists():
    kwp = load_kwp(str(plant_mapping_path), str(energy_coords_path), ds.sizes["plant"])
    print(f"Real kWp loaded: {int(np.sum(np.isfinite(kwp)))}/{ds.sizes['plant']} plants")

if CONFIG["apply_outlier_filter"]:
    ds, kwp, keep_mask = _filter_outlier_plants(
        ds,
        kwp,
        qs_daytime_threshold=CONFIG["qs_daytime_threshold"],
        min_n_valid_daytime=CONFIG["min_n_valid_daytime"],
    )
    print(f"After outlier filter: {ds.sizes['plant']} plants")


In [ ]:
seed_summary = []

for seed in CONFIG["seeds"]:
    ckpt = checkpoint_dir(CONFIG, seed)
    ckpt.mkdir(parents=True, exist_ok=True)
    name = run_name(CONFIG, seed)
    tags = list(CONFIG["wandb_tags"]) + [
        CONFIG["feature_set"],
        f"seq_len_{CONFIG['seq_len']}",
        f"seed_{seed}",
        f"pool_{CONFIG['bilstm_pooling']}",
    ]

    print("=" * 72)
    print(f"Seed {seed} | checkpoint -> {ckpt}")
    print(f"W&B run -> {name}")
    print("=" * 72)

    model, loss_history, val_loss_history, edge_index, edge_weight, pv_calibration = train(
        ds=ds,
        n_epochs=CONFIG["n_epochs"],
        lam=CONFIG["lam"],
        max_steps_per_epoch=CONFIG["max_steps_per_epoch"],
        kwp=kwp,
        early_stopping_patience=CONFIG["early_stopping_patience"],
        early_stopping_min_delta=CONFIG["early_stopping_min_delta"],
        peak_alpha=CONFIG["peak_alpha"],
        peak_gamma=CONFIG["peak_gamma"],
        peak_loss_weight=CONFIG["peak_loss_weight"],
        under_penalty=CONFIG["under_penalty"],
        calibration_kpi=CONFIG["calibration_kpi"],
        eta_max=CONFIG["eta_max"],
        batch_size=CONFIG["batch_size"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        num_workers=CONFIG["num_workers"],
        graph_max_dist_km=CONFIG["graph_max_dist_km"],
        d_model=CONFIG["d_model"],
        gat_dim=CONFIG["gat_dim"],
        gat_heads=CONFIG["gat_heads"],
        gat_layers=CONFIG["gat_layers"],
        dropout=CONFIG["dropout"],
        use_wandb=CONFIG["use_wandb"] and CONFIG["wandb_mode"] != "disabled",
        wandb_project=CONFIG["wandb_project"],
        wandb_entity=CONFIG["wandb_entity"],
        wandb_run_name=name,
        wandb_tags=tags,
        seq_len=CONFIG["seq_len"],
        checkpoint_dir=str(ckpt),
        use_bilstm=CONFIG["use_bilstm"],
        use_gat=CONFIG["use_gat"],
        bilstm_pooling=CONFIG["bilstm_pooling"],
        seed=seed,
    )

    best_val = min(val_loss_history) if val_loss_history else float("nan")
    best_epoch = val_loss_history.index(best_val) + 1 if val_loss_history else 0

    torch.save(model.state_dict(), ckpt / "model.pt")
    with open(ckpt / "loss_history.json", "w", encoding="utf-8") as f:
        json.dump({"train": loss_history, "val": val_loss_history, "best_epoch": best_epoch}, f, indent=2)
    with open(ckpt / "pv_calibration.json", "w", encoding="utf-8") as f:
        json.dump(jsonify(pv_calibration), f, indent=2)

    model_config = {
        "n_nodes": ds.sizes["plant"],
        "n_features": N_FEATURES,
        "seq_len": CONFIG["seq_len"],
        "d_model": CONFIG["d_model"],
        "gat_dim": CONFIG["gat_dim"],
        "gat_heads": CONFIG["gat_heads"],
        "gat_layers": CONFIG["gat_layers"],
        "dropout": CONFIG["dropout"],
        "use_bilstm": CONFIG["use_bilstm"],
        "use_gat": CONFIG["use_gat"],
        "bilstm_pooling": CONFIG["bilstm_pooling"],
        "seed": seed,
    }
    with open(ckpt / "model_config.json", "w", encoding="utf-8") as f:
        json.dump(jsonify(model_config), f, indent=2)

    training_config = dict(CONFIG)
    training_config.update({
        "checkpoint_dir": str(ckpt),
        "wandb_run_name": name,
        "wandb_tags_resolved": tags,
        "best_val_loss": best_val,
        "best_epoch": best_epoch,
    })
    with open(ckpt / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(jsonify(training_config), f, indent=2)

    seed_summary.append({
        "seed": seed,
        "wandb_run_name": name,
        "checkpoint_dir": str(ckpt),
        "best_val_loss": best_val,
        "best_epoch": best_epoch,
        "final_train_loss": loss_history[-1] if loss_history else float("nan"),
        "final_val_loss": val_loss_history[-1] if val_loss_history else float("nan"),
    })

seed_summary


In [ ]:
summary_path = Path(f"{project_path(CONFIG['checkpoint_dir_base'])}_multi_seed_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"config": jsonify(CONFIG), "runs": jsonify(seed_summary)}, f, indent=2)

print(f"Summary saved -> {summary_path}")

try:
    import pandas as pd
    display(pd.DataFrame(seed_summary))
except Exception:
    print(json.dumps(jsonify(seed_summary), indent=2))
